# 🧠 Mixture of Experts (MoE) Router - Groq API
Unit 2 Assignment

This notebook implements a Smart Customer Support Router using a Mixture of Experts architecture.

## 1️⃣ Install Dependencies

In [ ]:
!pip install groq python-dotenv

## 2️⃣ Import Libraries and Setup API

In [ ]:
import os
from groq import Groq
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

BASE_MODEL = "mixtral-8x7b-32768"


## 3️⃣ Define Expert Configurations

In [ ]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": """You are a Technical Support Expert.
Be precise, logical, and code-focused.
Explain errors clearly and provide corrected code when possible.
Avoid unnecessary fluff."""
    },
    "billing": {
        "system_prompt": """You are a Billing Support Specialist.
Be empathetic and professional.
Focus on subscriptions, refunds, and payment policies.
Provide clear next steps."""
    },
    "general": {
        "system_prompt": """You are a friendly and helpful general assistant.
Answer clearly and conversationally."""
    },
    "tool": {
        "system_prompt": "TOOL_HANDLER"
    }
}


## 4️⃣ Router Function

In [ ]:
def route_prompt(user_input: str) -> str:

    router_prompt = f"""
Classify the following user query into one of these categories:
[technical, billing, general, tool]

Rules:
- technical → bugs, errors, programming issues
- billing → payments, refunds, charges, subscriptions
- tool → requests for real-time data (e.g., current price of Bitcoin)
- general → anything else

Return ONLY the category word.

User Query:
{user_input}
"""

    response = client.chat.completions.create(
        model=BASE_MODEL,
        messages=[
            {"role": "system", "content": "You are a strict classifier."},
            {"role": "user", "content": router_prompt}
        ],
        temperature=0
    )

    category = response.choices[0].message.content.strip().lower()

    if category not in MODEL_CONFIG:
        return "general"

    return category


## 5️⃣ Tool Handler (Bonus)

In [ ]:
def get_bitcoin_price():
    # Mock data (replace with real API if needed)
    return "Bitcoin is currently trading at $63,500 (mock data)."


## 6️⃣ Orchestrator

In [ ]:
def process_request(user_input: str):

    category = route_prompt(user_input)
    print(f"[Router Decision]: {category}")

    if category == "tool":
        if "bitcoin" in user_input.lower():
            return get_bitcoin_price()
        else:
            return "Tool request recognized but not implemented."

    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    response = client.chat.completions.create(
        model=BASE_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ],
        temperature=0.7
    )

    return response.choices[0].message.content


## 7️⃣ Test the Router

In [ ]:
# Example test
query = "My python script is throwing an IndexError on line 5."
print(process_request(query))
